# Solutions — Advanced forms

One solution per exercise and per mini challenge, in lesson order.
Read these **after** you have tried. A solution you have not attempted teaches nothing.

Playground answers are prose plus the code to paste; the numbers quoted were measured while
writing the lessons.

### LESSON 78 — Exercise

In [ ]:
// L78 solution — move, validate, and cascading resets

const l78sRows = [
  { id: 1, name: "Ada", email: "ada@example.com" },
  { id: 2, name: "", email: "grace@example.com" },
  { id: 3, name: "Alan", email: "ada@example.com" },
];

// 1. move a row without mutating
function l78sMove(rows, id, direction) {
  const from = rows.findIndex((row) => row.id === id);
  const to = from + (direction === "up" ? -1 : 1);
  if (from === -1 || to < 0 || to >= rows.length) return rows;   // impossible: unchanged
  const copy = [...rows];
  [copy[from], copy[to]] = [copy[to], copy[from]];
  return copy;
}

console.log("moved 3 up:", l78sMove(l78sRows, 3, "up").map((r) => r.id));
console.log("moved 1 up:", l78sMove(l78sRows, 1, "up").map((r) => r.id), "(unchanged)");
console.log("same array returned?", l78sMove(l78sRows, 1, "up") === l78sRows);
console.log("original:", l78sRows.map((r) => r.id));

// 2. per-row and form-level validation
function l78sValidate(rows) {
  const seen = new Map();
  for (const row of rows) {
    const key = row.email.trim().toLowerCase();
    if (key) seen.set(key, (seen.get(key) ?? 0) + 1);
  }

  const perRow = rows.map((row) => {
    const errors = {};
    if (!row.name.trim()) errors.name = "Name is required";
    if (!row.email.includes("@")) errors.email = "That doesn't look like an email";
    else if (seen.get(row.email.trim().toLowerCase()) > 1) errors.email = "Duplicated in this form";
    return { id: row.id, errors };
  });

  const formErrors = perRow.some((r) => r.errors.email === "Duplicated in this form")
    ? ["Each attendee needs a different email"]
    : [];

  return { perRow, formErrors };
}

console.log("\nvalidation:", JSON.stringify(l78sValidate(l78sRows), null, 1));

// 3. cascading dependent resets
const l78sDependents = { country: ["region"], region: ["city"] };

function l78sReconcile(values, changed) {
  let next = { ...values };
  const queue = [...(l78sDependents[changed] ?? [])];
  while (queue.length) {
    const field = queue.shift();
    next[field] = "";
    queue.push(...(l78sDependents[field] ?? []));      // clear THEIR dependents too
  }
  return next;
}

const l78sValues = { country: "Italy", region: "Sicilia", city: "Palermo" };
console.log("\nbefore:", l78sValues);
console.log("country changed:", l78sReconcile({ ...l78sValues, country: "Germany" }, "country"));
console.log("region changed :", l78sReconcile({ ...l78sValues, region: "Lazio" }, "region"));

**Common mistake:** in part 3, clearing only the direct dependent. Change the country and the
region is cleared but the *city* still holds "Palermo" — a city in a region that is no longer
selected, in a country that no longer contains it. The queue is there so a change cascades all
the way down.

Also worth noticing in part 1: returning the *same array reference* when the move is impossible is
deliberate. In React that means `setRows(l78sMove(...))` with an impossible move bails out without
a re-render, because the state is `Object.is`-equal to what it already was (LESSON 27).

### LESSON 78 — Mini challenge

In [ ]:
// L78 solution — what index keys do to a form

const l78sTyped = [
  { id: "a", text: "first thing I typed" },
  { id: "b", text: "second thing I typed" },
  { id: "c", text: "third thing I typed" },
];

function l78sTrackByIndex(rows, removedIndex) {
  const remaining = rows.filter((_, index) => index !== removedIndex);
  return remaining.map((row, index) => ({ key: `index-${index}`, holds: row.text }));
}

function l78sTrackById(rows, removedId) {
  const remaining = rows.filter((row) => row.id !== removedId);
  return remaining.map((row) => ({ key: `id-${row.id}`, holds: row.text }));
}

console.log("before — keys and contents:");
for (const [index, row] of l78sTyped.entries()) {
  console.log(`  index-${index} / id-${row.id}  ->  "${row.text}"`);
}

console.log("\nremoving the middle row, keyed by INDEX:");
for (const row of l78sTrackByIndex(l78sTyped, 1)) console.log(`  ${row.key}  ->  "${row.holds}"`);

console.log("\nremoving the middle row, keyed by ID:");
for (const row of l78sTrackById(l78sTyped, "b")) console.log(`  ${row.key}  ->  "${row.holds}"`);

// React reuses the DOM node for an unchanged key.
//
// Under index keys, `index-1` existed before the removal (holding "second thing I typed") and
// still exists after it (now meant to hold "third thing I typed"). React sees the same key, keeps
// the same <input> node, and only updates what changed — so anything the DOM node owns and React
// does not, stays. The user watches the third row's text appear to jump up into the second row's
// box, along with the cursor position, the scroll offset, and any uncontrolled value.
//
// Why this is worse in a form than in a read-only list: in a list the wrong text renders and is
// immediately corrected on the next render, and nothing is lost. In a form the DOM node holds
// USER INPUT that exists nowhere else — and the user's own typing has now been attributed to the
// wrong row. They may not notice until they submit.

### LESSON 79 — Exercise

In [ ]:
// L79 solution — optional fields, all errors, and focus order

const l79sRequired = (label) => (value) =>
  value === undefined || value === null || String(value).trim() === "" ? `${label} is required` : null;
const l79sMin = (limit, message) => (value) => (Number(value) < limit ? message : null);
const l79sMatches = (pattern, message) => (value) =>
  value && !pattern.test(String(value)) ? message : null;

// 1. new rules — note that BOTH pass on an empty value
const l79sOneOf = (list, message) => (value) =>
  value && !list.includes(value) ? message : null;
const l79sMaxLength = (n, message) => (value) =>
  value && String(value).length > n ? message : null;

const l79sRules = {
  email: [l79sRequired("Email"), l79sMatches(/@/, "That doesn't look like an email")],
  age: [l79sRequired("Age"), l79sMin(18, "You must be 18 or over")],
  role: [l79sRequired("Role"), l79sOneOf(["admin", "editor", "viewer"], "Unknown role")],
  bio: [l79sMaxLength(100, "Keep the bio under 100 characters")],   // optional: no `required`
};

// 2. collect ALL failing rules
function l79sValidateAll(values, rules = l79sRules) {
  const errors = {};
  for (const [field, checks] of Object.entries(rules)) {
    const messages = checks.map((check) => check(values[field], values)).filter(Boolean);
    if (messages.length) errors[field] = messages;
  }
  return errors;
}

console.log("empty bio passes:", l79sValidateAll({ email: "a@b", age: 20, role: "admin", bio: "" }));
console.log("long bio fails  :", l79sValidateAll({ email: "a@b", age: 20, role: "admin", bio: "x".repeat(120) }));
console.log("everything wrong:", JSON.stringify(l79sValidateAll({ email: "ada", age: "16", role: "wizard", bio: "" })));

// a field that fails TWO rules at once — otherwise "collect them all" has nothing to show
const l79sPasswordRules = {
  password: [
    l79sRequired("Password"),
    l79sMatches(/.{8,}/, "At least 8 characters"),
    l79sMatches(/[0-9]/, "At least one digit"),
  ],
};
console.log("two failures   :", l79sValidateAll({ password: "abc" }, l79sPasswordRules));
console.log("first only     :", l79sValidateAll({ password: "abc" }, l79sPasswordRules).password[0]);

// Why show one message at a time anyway: a field with three messages under it is a wall of red
// that the user reads as "this is hopeless" rather than as instructions. Fixing the first problem
// usually changes which of the others still apply, so most of what you printed is speculative.
// Show the first, and show the next only if it survives the fix.

// 3. focus the first invalid field IN DISPLAY ORDER
function l79sFirstInvalid(errors, order) {
  return order.find((field) => errors[field]) ?? null;
}

const l79sOrder = ["email", "age", "role", "bio"];
const l79sErrs = l79sValidateAll({ email: "ada", age: "16", role: "wizard", bio: "" });
console.log("\nfocus:", l79sFirstInvalid(l79sErrs, l79sOrder));

// Why the order matters, and why Object.keys(errors) is not good enough: object key order follows
// how the rules object was written, not how the fields are laid out on screen — and it changes
// the moment someone reorders the form or the rules. Focusing "the first error" must mean the
// first one the USER would reach, or you scroll them past fields they have not seen.

### LESSON 79 — Mini challenge

In [ ]:
// L79 solution — an async check, with its two classic bugs

const l79sWait = (ms) => new Promise((r) => setTimeout(r, ms));
const l79sTaken = new Set(["ada", "grace"]);

// a lookup where "ada" is SLOW and "adam" is fast — so the stale answer arrives last
async function l79sLookup(username) {
  await l79sWait(username === "ada" ? 300 : 50);
  return l79sTaken.has(username) ? "taken" : "free";
}

function l79sMakeChecker(lookup) {
  return async function check(username) {
    try {
      return { status: await lookup(username) };
    } catch {
      return { status: "error" };
    }
  };
}

const l79sCheck = l79sMakeChecker(l79sLookup);

// --- bug 1: the stale answer ------------------------------------------------
let l79sField = { value: "", syncError: null, asyncStatus: "idle" };

async function l79sNaive(username) {
  l79sField = { ...l79sField, value: username, asyncStatus: "checking" };
  const { status } = await l79sCheck(username);
  l79sField = { ...l79sField, asyncStatus: status };       // no guard
}

const l79sSlow = l79sNaive("ada");
await l79sWait(20);
const l79sFast = l79sNaive("adam");
await Promise.all([l79sSlow, l79sFast]);
console.log("naive  — field value:", l79sField.value, "· status:", l79sField.asyncStatus, "  <- wrong: adam is free");

// --- fixed with L43's technique ---------------------------------------------
let l79sFixed = { value: "", syncError: null, asyncStatus: "idle" };
let l79sLatest = 0;

async function l79sGuarded(username) {
  const request = (l79sLatest += 1);
  l79sFixed = { ...l79sFixed, value: username, asyncStatus: "checking" };
  const { status } = await l79sCheck(username);
  if (request !== l79sLatest) {
    console.log(`   (ignoring the stale answer for "${username}")`);
    return;
  }
  l79sFixed = { ...l79sFixed, asyncStatus: status };
}

const l79sSlow2 = l79sGuarded("ada");
await l79sWait(20);
const l79sFast2 = l79sGuarded("adam");
await Promise.all([l79sSlow2, l79sFast2]);
console.log("fixed  — field value:", l79sFixed.value, "· status:", l79sFixed.asyncStatus, " <- correct");

// --- bug 2: submitting during a pending check -------------------------------
function l79sCanSubmitNaive(field) {
  return field.syncError === null;                       // says yes mid-check
}
function l79sCanSubmit(field) {
  return field.syncError === null && field.asyncStatus !== "checking" && field.asyncStatus !== "taken";
}

const l79sPending = { value: "ada", syncError: null, asyncStatus: "checking" };
console.log("\nnaive canSubmit while checking:", l79sCanSubmitNaive(l79sPending));
console.log("fixed canSubmit while checking:", l79sCanSubmit(l79sPending));

// Both answers are defensible:
//   Block the button while checking — simple, and the user never sends something you are about to
//   tell them is invalid. It costs a moment of a disabled button, which needs a visible reason
//   ("checking availability…") or it looks broken.
//   Let them submit and check on the server — better for a slow connection, and it is the honest
//   position anyway: the client's answer can be stale by the time the request lands, so the server
//   must re-check regardless. The client check is a courtesy, not a gate.
// What is NOT defensible is letting them submit while relying only on the client's stale answer.

### LESSON 80 — Exercise

**1. A select and a checkbox.** The select always contributes its current value. The checkbox
contributes `"on"` when ticked and **nothing at all** when unticked — the key is absent from the
FormData, so `formData.get("urgent")` is `null`. In a real form, normalise it in one place:
`const urgent = formData.get("urgent") === "on"`, or use `formData.has("urgent")`. Do not let a
missing key travel any further than the action's first three lines.

**2. Removing the `name`.** The field disappears from the submission entirely. `FormData` is built
from the form's named controls, exactly as an HTML form submission is — no `name`, no data. This
is the most common "my action gets an empty object" cause.

**3. An async action with a 3-second wait.** The form's fields stay interactive and the rest of the
page stays completely responsive — the submission is running in a Transition, so React is not
blocking the main thread waiting for it. What you do *not* get for free is a disabled submit button:
nothing stops the user pressing it again, which is precisely what LESSON 82's `pending` is for.

**4. Returning `{ ok: false, message }` instead of throwing.** Nothing appears. The action's return
value goes nowhere, because a plain `<form action>` discards it — React has no state to put it in.
That is exactly the gap `useActionState` fills, and it is worth feeling for one lesson before the
Hook arrives.

### LESSON 80 — Mini challenge

In [ ]:
// L80 solution — FormData for real

const l80Data = new FormData();
l80Data.append("title", "Release notes");
l80Data.append("tags", "react");
l80Data.append("tags", "forms");
l80Data.append("tags", "actions");
l80Data.append("empty", "");

// 1. a key that was never set
console.log("never set     :", l80Data.get("missing"));         // null
console.log("set to empty  :", JSON.stringify(l80Data.get("empty")));   // ""
console.log("different?    ", l80Data.get("missing") !== l80Data.get("empty"));

// 2. the same key several times
console.log("\nget('tags')   :", l80Data.get("tags"));          // the FIRST one only
console.log("getAll('tags'):", l80Data.getAll("tags"));

// 3. Object.fromEntries
console.log("\nfromEntries   :", Object.fromEntries(l80Data));  // the LAST one wins

// 4. a converter that keeps repeats
function l80ToObject(formData) {
  const out = {};
  for (const key of new Set(formData.keys())) {
    const all = formData.getAll(key);
    out[key] = all.length > 1 ? all : all[0];
  }
  return out;
}

console.log("l80ToObject   :", l80ToObject(l80Data));

// What (3) means for checkboxes sharing a name: a group of checkboxes all named "tags" is the
// normal HTML way to submit several values, and Object.fromEntries silently keeps only the last
// one. Every "only my last checkbox is saved" bug is this line. Use getAll for anything that can
// repeat — and note that `get` returning null for a missing key is how an unticked checkbox looks,
// which is a different thing from an empty string the user actually left blank.

### LESSON 81 — Exercise

**1. Swapping the parameters.** The action receives the previous state where it expects the form
data, so `formData.get` is not a function — the state object has no `get` method. The error points
at the line that calls it. This is why the argument order is worth memorising: with
`useActionState`, **state first, data second**.

**2. Returning a bare string.** It renders fine, and it is the wrong shape as soon as the form has
more than one thing to say. A form with per-field errors wants an object:

```js
return { ok: false, errors: { name: "Name is required" }, message: "Check the fields above." };
```

Because a string can hold only one message, and the moment you want "which field?" alongside "what
happened?" you would be parsing your own sentence. LESSON 67's rule about shapes that cannot
express an impossible state applies to what an action returns just as much as to props.

**3. Counting attempts without extra state.**

```jsx
async function saveName(previousState, formData) {
  const attempt = (previousState?.attempt ?? 0) + 1;
  const name = (formData.get("name") ?? "").trim();
  if (!name) return { ok: false, message: "Name is required.", attempt };
  return { ok: true, message: `Saved ${name}.`, attempt };
}

{state.attempt >= 3 && !state.ok && <p>You have tried {state.attempt} times — need help?</p>}
```

The counter lives in the state the action already threads through. A separate `useState` would be
a second source of truth that the action cannot see.

**4. Two forms, two Hooks.** They are entirely independent: each `useActionState` call has its own
state and its own pending flag. Sharing one Hook between two forms would give both the same state
object, so submitting one would change what the other displays — LESSON 64's rule exactly: a Hook
shares logic, not state, and calling it once and passing the result to two forms is the way to
*deliberately* share.

### LESSON 81 — Mini challenge

In [ ]:
// L81 solution — the action as an async reducer

async function l81Run(action, initialState, submissions) {
  let state = initialState;
  const log = [];
  for (const formData of submissions) {
    const before = state;
    state = await action(state, formData);
    log.push({ input: formData.name, from: before.message, to: state.message });
  }
  return { state, log };
}

const l81Wait = (ms) => new Promise((r) => setTimeout(r, ms));

async function l81SaveName(previousState, formData) {
  const name = (formData.name ?? "").trim();
  const saved = previousState.saved ?? [];
  await l81Wait(20);

  if (!name) return { ...previousState, ok: false, message: "Name is required." };
  if (saved.includes(name)) return { ...previousState, ok: false, message: `${name} is already saved.` };
  return { ok: true, message: `Saved ${name}.`, saved: [...saved, name] };
}

const l81Result = await l81Run(
  l81SaveName,
  { ok: null, message: "not submitted yet", saved: [] },
  [{ name: "" }, { name: "Ada" }, { name: "Ada" }, { name: "Grace" }],
);

for (const entry of l81Result.log) {
  console.log(`${JSON.stringify(entry.input).padEnd(9)} ${entry.from.padEnd(24)} -> ${entry.to}`);
}
console.log("\nfinal saved list:", l81Result.state.saved);

// What happens on a double submit in a real form: the second submission starts while the first is
// still running, and it is handed the state from BEFORE the first one returned — so a check like
// "is this name already saved?" passes twice and you save it twice. My l81Run cannot show this
// because it awaits each submission; that is the difference between a queue and what a browser
// actually does with two clicks.
//
// Which tool: all three, because they defend at different distances.
//   - useFormStatus / isPending disables the button: stops the ordinary double-click, costs
//     nothing, and is pure UI so it cannot be relied on.
//   - the pending flag also lets the action itself refuse to start twice.
//   - an idempotent server (a request id, a unique constraint) is the only one that actually
//     guarantees it, because the two clicks may come from two tabs, a flaky network retry, or a
//     user who reloaded. UI prevents the common case; the server prevents the case that matters.

### LESSON 82 — Exercise

**1. `status.data` in the label.**

```jsx
const { pending, data } = useFormStatus();
const name = data?.get("name");
return (
  <button type="submit" disabled={pending}>
    {pending ? `Saving ${name ? `"${name}"` : "…"}` : "Save"}
  </button>
);
```

Before any submission `data` is **`null`**, so it must be guarded — `data?.get(...)`. There is no
"empty FormData" state to fall back on.

**2. A `<Fieldset>` inside the form.**

```jsx
function Fieldset({ children }) {
  const { pending } = useFormStatus();
  return <fieldset disabled={pending}>{children}</fieldset>;
}
```

No props, and it works — which is the entire point of the Hook. `<fieldset disabled>` also
disables every control inside it, so one component freezes the whole form while it submits.

**3. Moving the button outside the form.** Two things break. The button no longer submits the form
at all (a `type="submit"` button submits the form it is *in*), and `pending` is always `false`,
because the Hook looks for a **parent** `<form>` and there is now none above it. The fix in real
HTML is the `form="formId"` attribute for the submit behaviour — but the Hook still sees no parent
form, so the pending state stays false. Keep the button inside.

**4. Two forms, one submitted.** The other button does **not** go pending. The Hook reports on the
nearest parent form in the React tree, not on "any submission on the page" — which is what makes a
shared `<SubmitButton />` safe to use in every form at once.

**Common mistake:** calling `useFormStatus` in the component that renders the form and concluding
the Hook is broken. Measured in experiment 35: `pending` is `false` there even mid-submission, and
the documentation says so. Use `useActionState`'s `isPending` in that component.

### LESSON 82 — Mini challenge

There is nothing to run — the deliverable is a component you keep. A good answer:

```jsx
import { useFormStatus } from "react-dom";

export function SubmitButton({ children, name, value }) {
  const { pending } = useFormStatus();
  return (
    <button type="submit" name={name} value={value} disabled={pending}>
      {pending ? "Working…" : children}
    </button>
  );
}
```

**1. Why `pending` is not a prop.** Because every caller would have to obtain it and pass it, which
means every form has to thread it down — and the whole reason the Hook exists is that the button
can read it from the form above it. A `pending` prop would also let a caller pass the *wrong*
value, which is a state that cannot currently exist.

**2. `children`, not `label`.** LESSON 67: content goes in `children`, so a caller can pass an icon
plus text, or a `<span>` for screen readers, without the component growing a prop for each. A
`label` string can only ever produce a string.

**3. Two submit buttons.** Give each a `name` and a `value` — `<button name="intent"
value="save">` and `<button name="intent" value="save-and-add">`. A submit button contributes its
own name/value pair to the FormData, so the action reads `formData.get("intent")` and the buttons
themselves read `status.data?.get("intent")` to know which of them is the one being pressed. That is
also how a single action serves two different outcomes without two forms.

**4. When the action fails.** The button should return to its normal label and stay enabled, so the
user can correct and retry — it should *not* try to display the error, which belongs next to the
field or above the form. `useFormStatus` does not know the action failed; it only knows the
submission finished. The Hook that knows is `useActionState`, whose returned state holds whatever
the action returned.

### LESSON 83 — Exercise

**1. `addOptimistic` outside an Action.** React warns that an optimistic state update was scheduled
outside of a transition (React's docs: *"If you call the setter outside an Action, React will show a
warning and the optimistic state will briefly render"*). On screen the message flashes in and
vanishes almost immediately, because there is no pending Action for it to live inside — the
optimistic value only exists for the duration of a Transition.

**2. Two messages in flight with `id: "optimistic"`.** Both optimistic entries carry the same key,
so React warns about duplicate keys and the list behaves unpredictably — LESSON 20 in a new place.
The fix is a unique temporary id per entry:

```jsx
const [optimisticMessages, addOptimistic] = useOptimistic(
  messages,
  (current, newText) => [
    ...current,
    { id: `optimistic-${newText}-${current.length}`, text: newText, sending: true },
  ],
);
```

A `crypto.randomUUID()` works too — the requirement is only that it is unique among the entries
rendered at the same moment.

**3. Keeping a failed message visible with a retry.** It must move into **real state**. The first
Key Note is the reason: optimistic state exists *only while an Action is running*, so anything that
must outlive the Action cannot live there. On failure, add the message to real state with a
`status: "failed"` flag and render a retry button beside it.

**4. A 50 ms delay.** Almost certainly not worth it: the confirmed state arrives in about the time
it takes the screen to repaint, so the optimistic entry is never really seen. What to measure: the
*real* latency of that action for real users — the p50 and p95 round-trip, not localhost. Optimistic
UI earns its complexity at a few hundred milliseconds and up, which is where a user starts to feel
the wait.

### LESSON 83 — Mini challenge

In [ ]:
// L83 solution — when to bet

function l83ShouldBeOptimistic({ name, successRate, failureCost, reversible }) {
  if (failureCost === "high") {
    return { name, optimistic: false, why: "a wrong optimistic result costs the user something real" };
  }
  if (!reversible) {
    return { name, optimistic: false, why: "irreversible: confirm it, do not predict it" };
  }
  if (successRate < 0.95) {
    return { name, optimistic: false, why: `fails too often (${Math.round((1 - successRate) * 100)}% of the time)` };
  }
  return { name, optimistic: true, why: "very likely to succeed, cheap and reversible if not" };
}

const l83Actions = [
  { name: "send a chat message", successRate: 0.99, failureCost: "low", reversible: true },
  { name: "like a post", successRate: 0.995, failureCost: "low", reversible: true },
  { name: "book the last seat", successRate: 0.97, failureCost: "high", reversible: false },
  { name: "pay an invoice", successRate: 0.98, failureCost: "high", reversible: false },
  { name: "rename a file", successRate: 0.99, failureCost: "low", reversible: true },
  { name: "delete an account", successRate: 0.99, failureCost: "low", reversible: false },
];

for (const action of l83Actions) {
  const verdict = l83ShouldBeOptimistic(action);
  console.log(`${verdict.name.padEnd(22)} ${(verdict.optimistic ? "OPTIMISTIC" : "wait").padEnd(12)} ${verdict.why}`);
}

// 1. Booking the last seat and paying an invoice both succeed ~98% of the time and still must not
//    be optimistic. What separates them is failureCost, not the rate: the 2% case leaves the user
//    believing they have a seat or that a bill is paid, and acting on it — telling a colleague,
//    closing the tab. A 2% chance of a wrong belief about money or scarcity is not a UI trade-off.
//
// 2. Deleting an account: the failure is cheap, but it is irreversible, so optimistic UI is the
//    wrong instrument entirely — not because the prediction is risky but because this action wants
//    MORE friction, not less. Show a confirmation, do the work, and report the real result.
//
// 3. Where successRate comes from: your own telemetry — the ratio of failed to total calls for that
//    endpoint, ideally split by network conditions. Before you have that number, assume you do not
//    know it and reserve optimistic UI for the cases where even a frequent failure is harmless (a
//    like, a message you can resend). "Probably fine" is not a measurement, and this is the one
//    place in the course where guessing costs the user rather than the developer.